# 7.12 Sequence Modeling Foundations

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook introduces the core ideas behind sequence models before getting into particular architectures such as LSTMs, GRUs, and transformers.


## 7.12.1 Sequences and order sensitivity

### Why It Matters
Unlike tabular rows, sequences change meaning when the order changes. Two sentences with the same tokens in different orders should not be treated as identical inputs.


In [ ]:
import pandas as pd
from collections import Counter

sentence_a = ["not", "good"]
sentence_b = ["good", "not"]

bigrams_a = Counter(zip(sentence_a, sentence_a[1:]))
bigrams_b = Counter(zip(sentence_b, sentence_b[1:]))

pd.DataFrame({
    "sentence": ["not good", "good not"],
    "bag_of_words": [sorted(sentence_a), sorted(sentence_b)],
    "bigrams": [dict(bigrams_a), dict(bigrams_b)],
    "same_bag_of_words": [sorted(sentence_a) == sorted(sentence_b)] * 2,
})


### Interpretation
This is the motivating reason for sequence-aware architectures: order carries information.


## 7.12.2 Token and step representation

### Why It Matters
Before a model can read a sequence, discrete tokens must become integer indices or vectors. That mapping is the first layer of sequence preprocessing.


In [ ]:
import torch
from torch import nn

vocab = {token: idx for idx, token in enumerate(["<pad>", "i", "like", "neural", "nets"])}
sentence = ["i", "like", "neural", "nets"]
indices = torch.tensor([vocab[token] for token in sentence])
embed = nn.Embedding(len(vocab), 4)
embedded = embed(indices)

{
    "indexed_sentence": indices.tolist(),
    "embedded_shape": list(embedded.shape),
    "first_token_vector": embedded[0].detach().round(decimals=3).tolist(),
}


### Interpretation
Integer token ids are usually the bridge between text preprocessing and learned embeddings.


## 7.12.3 Recurrent intuition

### Why It Matters
A recurrent model processes one time step at a time while carrying a hidden state forward. This cell uses a toy recurrence to show the running-state idea.


In [ ]:
import torch

sequence = torch.tensor([1.0, -0.5, 2.0, 0.5])
hidden = torch.tensor(0.0)
states = []
for step in sequence:
    hidden = 0.6 * hidden + step
    states.append(round(float(hidden.item()), 3))
states


### Interpretation
Each new state mixes the current token information with memory from earlier steps.


## 7.12.4 Hidden state

### Why It Matters
The hidden state is the model’s running summary of what it has seen so far. Watching it evolve makes the memory idea more concrete.


In [ ]:
import torch
from torch import nn

rnn = nn.RNN(input_size=1, hidden_size=2, batch_first=True)
x = torch.tensor([[[1.0], [0.0], [2.0], [1.0]]])
output, hidden = rnn(x)
{"all_hidden_states_shape": list(output.shape), "final_hidden_state": hidden.detach().squeeze(0).round(decimals=3).tolist()}


### Interpretation
The final hidden state is one compressed summary of the full prefix seen by the model.


## 7.12.5 Teacher forcing intuition

### Why It Matters
Sequence decoders are often trained by feeding the true previous token rather than the model’s own last guess. That is called teacher forcing.


In [ ]:
import torch

start = {"<start>": 0, "the": 1, "cat": 2, "sat": 3}
target_sequence = torch.tensor([start["the"], start["cat"], start["sat"]])
teacher_forced_inputs = torch.tensor([start["<start>"], *target_sequence[:-1].tolist()])
model_generated_inputs = torch.tensor([start["<start>"], start["the"], start["the"]])

{
    "target_sequence": target_sequence.tolist(),
    "teacher_forced_inputs": teacher_forced_inputs.tolist(),
    "free_running_inputs_example": model_generated_inputs.tolist(),
}


### Interpretation
Teacher forcing makes training easier, but it also means training-time inputs can differ from inference-time inputs.


## 7.12.6 Prepare sequential data in PyTorch

### Why It Matters
A practical sequence workflow needs padded batches and sequence tensors with a clear batch and time dimension.


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

seqs = [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6])]
padded = pad_sequence(seqs, batch_first=True, padding_value=0)
lengths = [len(s) for s in seqs]
{"padded_batch": padded.tolist(), "shape": list(padded.shape), "lengths": lengths}


### Interpretation
Padding is how variable-length sequences are turned into tensors that can move through a batched model.
